# 04 · CatBoost

Gradient boosting con árboles simétricos y *ordered boosting*. Las categóricas de baja cardinalidad se le pasan nativas; las de alta cardinalidad ya vienen como target encoding.

Usa las features generadas por `00_features.ipynb`. Validación: 5 folds estratificados, los mismos para todos los modelos; las features que dependen de la etiqueta están recalculadas dentro de cada fold.

In [1]:
import os, time, numpy as np, pandas as pd, warnings
import features as F
warnings.filterwarnings("ignore")
os.makedirs("models", exist_ok=True)
y = F.load_y(); folds = F.cv_folds(y)
from catboost import CatBoostClassifier

## Modelo

In [2]:
PARAMS = dict(iterations=800, learning_rate=0.08, depth=8, l2_leaf_reg=3, max_ctr_complexity=1, one_hot_max_size=32,
              thread_count=8, verbose=0, allow_writing_files=False)
SEEDS = [42, 43, 44]   # se promedian varias semillas para reducir la varianza del modelo

def fit_predict(A, ya, B, return_model=False):
    cats = [c for c in A.columns if str(A[c].dtype) == "category"]
    A, B = A.copy(), B.copy()
    A[cats], B[cats] = A[cats].astype(str), B[cats].astype(str)
    models = [CatBoostClassifier(**PARAMS, random_seed=s).fit(A, ya, cat_features=cats) for s in SEEDS]
    p = np.mean([m.predict_proba(B)[:, 1] for m in models], axis=0)
    return (p, models) if return_model else p

## Validación cruzada

In [3]:
oof = np.zeros(len(y)); t0 = time.time()
for f, (a, b) in enumerate(folds):
    A, B = F.load_fold(f)
    oof[b] = fit_predict(A, y[a], B)
    print(f"fold {f}: accuracy {((oof[b] > .5) == y[b]).mean():.4f}  ({time.time() - t0:.0f}s)")
print(f"\nAccuracy CV: {((oof > .5) == y).mean():.4f}")
np.save(f"{F.CACHE}/oof_catboost.npy", oof)

fold 0: accuracy 0.9503  (531s)


fold 1: accuracy 0.9484  (962s)


fold 2: accuracy 0.9490  (1336s)


fold 3: accuracy 0.9464  (1773s)


fold 4: accuracy 0.9466  (2678s)

Accuracy CV: 0.9481


## Modelo final (todo train) y predicción de test

In [4]:
A, B = F.load_full()
proba, model = fit_predict(A, y, B, return_model=True)
for s, m in zip(SEEDS, model):
    m.save_model(f"models/catboost_seed{s}.cbm")
np.save(f"{F.CACHE}/test_catboost.npy", proba)
F.save_submission(proba, "catboost")["condition"].value_counts(normalize=True)

condition
new     0.527933
used    0.472067
Name: proportion, dtype: float64